# MHC II patient classification — LUAD cancer cells

**Goal:** Classify LUAD patients into MHC II High and MHC II Low groups based
on single-cell RNA sequencing data from primary tumor cancer cells. This analysis
is motivated by IHC-based observations in which a pathologist identified patients
whose tumor cells express MHC II protein and patients whose tumor cells do not.
The goal here is to recapitulate and extend this classification in publicly
available scRNA-seq data, enabling downstream transcriptomic comparisons between
MHC II High and Low tumors at scale.

**Gene set rationale:** Classification uses the core MHC II heterodimer genes
(HLA-DRA, HLA-DRB1, HLA-DQA1, HLA-DQB1, HLA-DPA1, HLA-DPB1) plus CD74,
which is essential for MHC II peptide loading. This set approximates the
molecules targeted by anti-MHC II antibodies and captures the core antigen
presentation machinery without including regulatory genes (CIITA) or accessory
molecules (HLA-DMA, HLA-DOB) that may add noise.

**Input:** Salcher et al. pan-cancer scRNA-seq atlas (`local.h5ad`), subset to
LUAD primary tumor cancer cells.

**Output:** `outputs/processed/luad_mhc2_classified.h5ad` — full LUAD AnnData
object with `MHC2_clustering` added to `obs` for all samples. Samples without
sufficient cancer cells are labeled 'Excluded from Clustering'.

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell analysis
import scanpy as sc

# clustering and statistics
from sklearn.cluster import AgglomerativeClustering
from scipy.stats import zscore
from scipy.cluster.hierarchy import dendrogram, linkage

# utilities
from tqdm import tqdm
from pathlib import Path
import yaml

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 20})

# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in UMAPs

In [ ]:
# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
atlas_path = cfg['datasets']['salcher_atlas']['raw']

# output paths
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']
fig_out   = fig_dir / 'figure2-supp'
table_out = table_dir / 'figure2-supp'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# load full atlas
adata = sc.read_h5ad(atlas_path)
print(f'Loaded atlas: {adata.n_obs:,} cells')

In [ ]:
# subset to LUAD
adenodata = adata[adata.obs['disease'] == 'lung adenocarcinoma'].copy()
print(f'LUAD cells: {adenodata.n_obs:,}')

# subset to primary tumor cancer cells for clustering
tumor_data = adenodata[
    (adenodata.obs['origin'] == 'tumor_primary') &
    (adenodata.obs['ann_fine'] == 'Cancer cells')
].copy()
print(f'Primary tumor cancer cells: {tumor_data.n_obs:,}')
print(f'Unique samples: {tumor_data.obs["sample"].nunique()}')

## Gene list

Core MHC II heterodimer genes plus CD74. This set was chosen to approximate
the molecules targeted by anti-MHC II antibodies and to capture the core
antigen presentation machinery. Regulatory (CIITA) and accessory (HLA-DMA,
HLA-DOB) genes are excluded to reduce noise in the clustering.

In [ ]:
# core MHC II genes used for patient classification
classification_genes = [
    'HLA-DRA', 'HLA-DRB1',
    'HLA-DQA1', 'HLA-DQB1',
    'HLA-DPA1', 'HLA-DPB1',
    'CD74',
]

## Per-sample mean expression

Single-cell expression is aggregated to sample-level means before clustering.
This removes within-sample cell-to-cell variability and gives one value per
patient per gene, which is the appropriate unit for patient classification.

In [ ]:
# aggregate to sample-level mean expression
samples = list(tumor_data.obs['sample'].unique())
cols    = classification_genes + ['number_of_cells']
bulk_df = pd.DataFrame(columns=cols)

for sample in tqdm(samples):
    patient = tumor_data[tumor_data.obs['sample'] == sample]
    values  = []
    for gene in classification_genes:
        gene_mask = patient.var['feature_name'] == gene
        if gene_mask.sum() == 0:
            values.append(0.0)
            continue
        x = patient[:, gene_mask].X
        values.append(float(x.toarray().mean()) if hasattr(x, 'toarray') else float(x.mean()))
    values.append(len(patient.obs))
    bulk_df.loc[sample] = values

print(f'Samples aggregated: {len(bulk_df)}')
bulk_df

## Clustering

Z-score normalization is applied per gene before clustering so that genes
with different expression scales contribute equally. Agglomerative clustering
with Ward linkage divides samples into two groups — MHC II High and MHC II Low.

In [ ]:
# z-score normalization per gene
df_z_score = pd.DataFrame(
    zscore(bulk_df.iloc[:, :-1]),
    index=bulk_df.index,
    columns=bulk_df.iloc[:, :-1].columns,
)

# dendrogram — QC to confirm two clear clusters before committing to k=2
linkage_data = linkage(df_z_score, method='ward', metric='euclidean')
dendrogram(linkage_data, color_threshold=25)
plt.title('Hierarchical clustering — MHC II genes in LUAD cancer cells')
plt.xticks([])
plt.xlabel('Samples')
plt.tight_layout()
plt.show()

In [ ]:
# agglomerative clustering — k=2 (MHC II High vs Low)
clustering_model = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
clustering_model.fit(df_z_score)

# map numeric labels to string labels
# 0 = MHC class II High, 1 = MHC class II Low
# verify this mapping by checking which cluster has higher mean HLA-DRA
label_map = {0: 'MHC class II High', 1: 'MHC class II Low'}

data_labels = pd.Series(clustering_model.labels_, index=df_z_score.index)

# confirm label orientation — cluster with higher mean HLA-DRA should be High
mean_by_cluster = bulk_df.groupby(data_labels)['HLA-DRA'].mean()
print('Mean HLA-DRA by cluster:')
print(mean_by_cluster)
print('\nCluster 0 should have higher HLA-DRA to confirm label_map is correct')

In [ ]:
# apply string labels
data_labels = data_labels.map(label_map)
print(data_labels.value_counts())

In [ ]:
# QC clustermap — visualize clustering result
# colors match paper-wide palette: MHC II High = orange (#FF8811), Low = purple (#462255)
sns.set(font_scale=1.6)
sns.set_style('ticks')

lut        = {'MHC class II High': '#FF8811', 'MHC class II Low': '#462255'}
row_colors = data_labels.map(lut)

g = sns.clustermap(
    df_z_score,
    row_colors=row_colors,
    cmap='vlag', center=0,
    method='ward', metric='euclidean',
    vmin=-2, vmax=2,
    figsize=(6, 4),        # narrow and tall — matches fig S4a layout
    dendrogram_ratio=0.2,  # keep dendrogram compact
    colors_ratio=0.03,      # thin color bar
    xticklabels=True,
    yticklabels=False,      # too many samples to label
)
plt.setp(g.ax_heatmap.xaxis.get_majorticklabels(), rotation=90, fontsize=16)
#g.fig.suptitle('MHC II patient clustering', y=1.02, fontsize=12)
g.fig.savefig(fig_out / 'supp_figure4a_mhc2_classification_clustermap.pdf', bbox_inches='tight', dpi=300)
plt.show()

## Add classification to full LUAD object

The MHC II High/Low label is added to all cells in the full LUAD object.
Samples without primary tumor cancer cells (and therefore not included in
the clustering) are labeled 'Excluded from Clustering'.

In [ ]:
# map sample-level labels to all cells in the full LUAD object
adenodata.obs['MHC2_clustering'] = (
    adenodata.obs['sample']
    .map(data_labels)
    .fillna('Excluded from Clustering')
)

print(adenodata.obs['MHC2_clustering'].value_counts())

In [ ]:
# save classified LUAD object
out_path = repo_root / cfg['outputs']['processed'] / 'luad_mhc2_classified.h5ad'
out_path.parent.mkdir(parents=True, exist_ok=True)
adenodata.write(out_path)
print(f'Saved → {out_path}')

In [ ]:
# save patient-level classification table
classification_table = pd.DataFrame({
    'sample':           data_labels.index,
    'MHC2_clustering':  data_labels.values,
}).merge(
    bulk_df.reset_index().rename(columns={'index': 'sample'}),
    on='sample',
)
classification_table.to_csv(
    table_out / 'mhc2_patient_classification.csv', index=False
)
print(f'Saved classification table → {table_out / "mhc2_patient_classification.csv"}')
classification_table